In [41]:
import pandas as pd
df = pd.read_csv('raw_legacy_banking_data.csv')
df.head(10)

,Transaction_Timestamp,Customer_ID,KYC_Approval_Date,Customer_Metadata,Fraud_Score,Account_Details
0,2026-06-01 07:21:00,CUST-65333,45734,"{""risk_tier"": ""Low"", ""branch"": ""Mumbai_Hub""} -...",57.92,SAVINGS_Dormant_EUR
1,2026-06-01 08:41:00,CUST-31992,24-10-2025,"{""risk_tier"": ""High"", ""branch"": ""Delhi_West""} ...",26.60,SAVINGS-Dormant-USD
2,2026-06-01 23:40:00,CUST-73156,06-02-2025,"{""risk_tier"": ""Medium"", ""branch"": ""Mumbai_Hub""...",NaN,SAVINGS-Active-INR
3,2026-06-01 15:19:00,CUST-20458,03/14/2025,"{""risk_tier"": ""High"", ""branch"": ""Delhi_West""} ...",NaN,CREDIT|Frozen|INR
4,2026-06-01 03:25:00,CUST-50189,01-12-2025,Risk Tier is High assigned at Delhi_West - Ver...,77.95,CREDIT_Frozen_INR
5,2026-06-01 17:21:00,CUST-92447,45899,"{""risk_tier"": ""Medium"", ""branch"": ""Kota_Inland...",26.04,SAVINGS_Dormant_USD
6,2026-06-01 20:23:00,CUST-10726,24-01-2025,"{""risk_tier"": ""High"", ""branch"": ""Jaipur_Main""}...",NaN,SAVINGS-Active-USD
7,2026-06-01 14:13:00,CUST-58916,46017,"{""risk_tier"": ""High"", ""branch"": ""Kota_Inland""}...",15.03,CURRENT_Frozen_EUR
8,2026-06-01 19:42:00,CUST-56421,08-03-2025,"{""risk_tier"": ""Medium"", ""branch"": ""Jaipur_Main...",0.14,CURRENT-Frozen-EUR
9,2026-06-01 10:09:00,CUST-72493,45746,"{""risk_tier"": ""Medium"", ""branch"": ""Kota_Inland...",79.97,SAVINGS|Dormant|INR


In [42]:
# Cleaning & standardizing dates
def clean_kyc_dates(date_val):
    if str(date_val).isdigit():
        return pd.to_datetime('1899-12-30') + pd.to_timedelta(int(date_val), unit='D')
    else:
        return pd.to_datetime(date_val, dayfirst=True, errors='coerce')

# Applying the cleaning function to create a new date column:
df['KYC_Approval_Date_Clean'] = df['KYC_Approval_Date'].apply(clean_kyc_dates)

#Removing the original messy date column:
df.drop(columns=['KYC_Approval_Date'], inplace=True)

df.head(10)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_11080\3132695263.py:6: UserWarning: Parsing dates in %m/%d/%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(date_val, dayfirst=True, errors='coerce')


,Transaction_Timestamp,Customer_ID,Customer_Metadata,Fraud_Score,Account_Details,KYC_Approval_Date_Clean
0,2026-06-01 07:21:00,CUST-65333,"{""risk_tier"": ""Low"", ""branch"": ""Mumbai_Hub""} -...",57.92,SAVINGS_Dormant_EUR,2025-03-18
1,2026-06-01 08:41:00,CUST-31992,"{""risk_tier"": ""High"", ""branch"": ""Delhi_West""} ...",26.60,SAVINGS-Dormant-USD,2025-10-24
2,2026-06-01 23:40:00,CUST-73156,"{""risk_tier"": ""Medium"", ""branch"": ""Mumbai_Hub""...",NaN,SAVINGS-Active-INR,2025-02-06
3,2026-06-01 15:19:00,CUST-20458,"{""risk_tier"": ""High"", ""branch"": ""Delhi_West""} ...",NaN,CREDIT|Frozen|INR,2025-03-14
4,2026-06-01 03:25:00,CUST-50189,Risk Tier is High assigned at Delhi_West - Ver...,77.95,CREDIT_Frozen_INR,2025-12-01
5,2026-06-01 17:21:00,CUST-92447,"{""risk_tier"": ""Medium"", ""branch"": ""Kota_Inland...",26.04,SAVINGS_Dormant_USD,2025-08-30
6,2026-06-01 20:23:00,CUST-10726,"{""risk_tier"": ""High"", ""branch"": ""Jaipur_Main""}...",NaN,SAVINGS-Active-USD,2025-01-24
7,2026-06-01 14:13:00,CUST-58916,"{""risk_tier"": ""High"", ""branch"": ""Kota_Inland""}...",15.03,CURRENT_Frozen_EUR,2025-12-26
8,2026-06-01 19:42:00,CUST-56421,"{""risk_tier"": ""Medium"", ""branch"": ""Jaipur_Main...",0.14,CURRENT-Frozen-EUR,2025-03-08
9,2026-06-01 10:09:00,CUST-72493,"{""risk_tier"": ""Medium"", ""branch"": ""Kota_Inland...",79.97,SAVINGS|Dormant|INR,2025-03-30


In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 6 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Transaction_Timestamp    5000 non-null   object        
 1   Customer_ID              5000 non-null   object        
 2   Customer_Metadata        5000 non-null   object        
 3   Fraud_Score              4260 non-null   float64       
 4   Account_Details          5000 non-null   object        
 5   KYC_Approval_Date_Clean  5000 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(1), object(4)
memory usage: 234.5+ KB


In [44]:
import json
def extract_risk(text):
    # converting cell contents to text:
    text = str(text)

    # Scenario A - text contains a JSON object:
    if '{risk_tier"' in text:
        clean_json_string = text.split('}')[0] + '}'
        dictionary_data = json.loads(clean_json_string)
        return dictionary_data('risk_tier')
    else:
        if "High" in text:
            return "High"
        elif "Medium" in text:
            return "Medium"
        elif "Low" in text:
            return "Low"
        else:
            return "Unknown"

# applying the function to create our clean risk column
df['Risk_Tier_Clean'] = df['Customer_Metadata'].apply(extract_risk)

#Removing the original Metadata column:
df.drop(columns=['Customer_Metadata'], inplace=True)

df.head(5)

,Transaction_Timestamp,Customer_ID,Fraud_Score,Account_Details,KYC_Approval_Date_Clean,Risk_Tier_Clean
0,2026-06-01 07:21:00,CUST-65333,57.92,SAVINGS_Dormant_EUR,2025-03-18,Low
1,2026-06-01 08:41:00,CUST-31992,26.60,SAVINGS-Dormant-USD,2025-10-24,High
2,2026-06-01 23:40:00,CUST-73156,NaN,SAVINGS-Active-INR,2025-02-06,Medium
3,2026-06-01 15:19:00,CUST-20458,NaN,CREDIT|Frozen|INR,2025-03-14,High
4,2026-06-01 03:25:00,CUST-50189,77.95,CREDIT_Frozen_INR,2025-12-01,High


In [45]:
# Cleaning the logical duplicates in the dataset:

#converting to datetime object:
df['Transaction_Timestamp'] = pd.to_datetime(df['Transaction_Timestamp'])

# Sort the table by Customer ID, then by Time (oldest to newest)
df = df.sort_values (by=['Customer_ID', 'Transaction_Timestamp'], ascending=[True, True])

#recording the row counts before deleting duplicates:
rows_before = len(df)

# removing duplicates and keeping the last:
df.drop_duplicates(subset=['Customer_ID'], keep='last', inplace=True)

#recording the row counts after deleting duplicates:
rows_after = len(df)

# Printing the impact:
print(f"Rows before removing duplicates: {rows_before}")
print(f"Rows after removing duplicates: {rows_after}")
print(f"Number of duplicates removed: {rows_before - rows_after}")

# check the clean data:
df.head()

Rows before removing duplicates: 5000
Rows after removing duplicates: 2413
Number of duplicates removed: 2587


,Transaction_Timestamp,Customer_ID,Fraud_Score,Account_Details,KYC_Approval_Date_Clean,Risk_Tier_Clean
598,2026-06-01 11:04:00,CUST-10053,27.87,SAVINGS|Dormant|INR,2025-11-26,Medium
1354,2026-06-01 16:08:00,CUST-10074,-999.00,SAVINGS-Frozen-INR,2025-12-03,Medium
2764,2026-06-01 01:25:00,CUST-10150,76.61,CURRENT_Active_USD,2025-11-21,Medium
3920,2026-06-01 09:18:00,CUST-10205,82.80,CREDIT|Active|INR,2025-07-17,High
1765,2026-06-01 09:17:00,CUST-10221,73.58,SAVINGS|Active|EUR,2025-03-17,Low


In [46]:
# Handling NULL & negative values in Fraud_Score column:
# Force the column into numbers.
df['Fraud_Score_Clean'] = pd.to_numeric(df['Fraud_Score'], errors='coerce')

# Still have the -999 - (which Python thinks is a valid number). 
# Let's locate any score less than 0 and turn it into a true NaN with NUMPY:
import numpy as np
df.loc[df['Fraud_Score_Clean'] < 0, 'Fraud_Score_Clean'] = np.nan

# checking the missing values in the Fraud_Score_Clean column:
missing_count = df['Fraud_Score_Clean'].isna().sum()
print(f"Number of missing values in Fraud_Score_Clean: {missing_count}")

# removing the original Fraud_Score column:
df.drop(columns=['Fraud_Score'], inplace=True)

#Checking the clean data:
df.head()

Number of missing values in Fraud_Score_Clean: 474


,Transaction_Timestamp,Customer_ID,Account_Details,KYC_Approval_Date_Clean,Risk_Tier_Clean,Fraud_Score_Clean
598,2026-06-01 11:04:00,CUST-10053,SAVINGS|Dormant|INR,2025-11-26,Medium,27.87
1354,2026-06-01 16:08:00,CUST-10074,SAVINGS-Frozen-INR,2025-12-03,Medium,NaN
2764,2026-06-01 01:25:00,CUST-10150,CURRENT_Active_USD,2025-11-21,Medium,76.61
3920,2026-06-01 09:18:00,CUST-10205,CREDIT|Active|INR,2025-07-17,High,82.80
1765,2026-06-01 09:17:00,CUST-10221,SAVINGS|Active|EUR,2025-03-17,Low,73.58


In [47]:
# Standardize the delimiters. Replace pipes and dashes with underscores.
df['Account_Details'] = df['Account_Details'].str.replace('|', '_', regex=False).str.replace('-', '_', regex=False)

# Split the column at the underscore using The expand=True
df[['Account_type', 'Account_status', 'Currecy']] = df['Account_Details'].str.split('_', expand=True)

# removing the original Account_details column:
df.drop(columns=['Account_Details'], inplace=True)

df.head()

,Transaction_Timestamp,Customer_ID,KYC_Approval_Date_Clean,Risk_Tier_Clean,Fraud_Score_Clean,Account_type,Account_status,Currecy
598,2026-06-01 11:04:00,CUST-10053,2025-11-26,Medium,27.87,SAVINGS,Dormant,INR
1354,2026-06-01 16:08:00,CUST-10074,2025-12-03,Medium,NaN,SAVINGS,Frozen,INR
2764,2026-06-01 01:25:00,CUST-10150,2025-11-21,Medium,76.61,CURRENT,Active,USD
3920,2026-06-01 09:18:00,CUST-10205,2025-07-17,High,82.80,CREDIT,Active,INR
1765,2026-06-01 09:17:00,CUST-10221,2025-03-17,Low,73.58,SAVINGS,Active,EUR


In [ ]:
# Saving the cleaned data to a new Excel file:

# Reorder our columns to make the final table look professional
column_order = [
    'Transaction_Timestamp', 'Customer_ID', 'KYC_Approval_Date_Clean', 
    'Risk_Tier_Clean', 'Fraud_Score_Clean', 'Account_Type', 'Account_Status', 'Currency'
]
df = df[column_order]

# Remove the timezones from the dates
df['Transaction_Timestamp'] = df['Transaction_Timestamp'].dt.tz_localize(None)
df['KYC_Approval_Date_Clean'] = df['KYC_Approval_Date_Clean'].dt.tz_localize(None)

# Export to Excel without the random index numbers
df.to_excel('python_cleaned_data.xlsx', index=False, sheet_name='Python_Cleaned_Data')

print("Dataset successfully exported to Excel!")

In [51]:
### COMBINING ALL STEPS INTO A SINGLE PIPELINE and Exporting to Excel:

import pandas as pd
import numpy as np
import json

# 1. LOAD RAW DATA FRESH (Bypasses all memory issues)
df = pd.read_csv('raw_banking_data.csv')

# 2. CLEAN DATES
def clean_kyc_dates(date_val):
    val = str(date_val)
    if val.isdigit():
        return pd.to_datetime('1899-12-30') + pd.to_timedelta(int(val), unit='D')
    else:
        return pd.to_datetime(val, dayfirst=True, errors='coerce')
df['KYC_Approval_Date_Clean'] = df['KYC_Approval_Date'].apply(clean_kyc_dates)
df.drop(columns=['KYC_Approval_Date'], inplace=True)

# 3. CLEAN JSON
def extract_risk(text):
    text = str(text)
    if '{"risk_tier"' in text:
        return json.loads(text.split('}')[0] + '}')['risk_tier']
    else:
        if 'High' in text: return 'High'
        elif 'Medium' in text: return 'Medium'
        elif 'Low' in text: return 'Low'
        else: return 'Unknown'
df['Risk_Tier_Clean'] = df['Customer_Metadata'].apply(extract_risk)
df.drop(columns=['Customer_Metadata'], inplace=True)

# 4. PURGE DUPLICATES
df['Transaction_Timestamp'] = pd.to_datetime(df['Transaction_Timestamp'])
df = df.sort_values(by=['Customer_ID', 'Transaction_Timestamp'])
df.drop_duplicates(subset=['Customer_ID'], keep='last', inplace=True)

# 5. HANDLE NULLS
df['Fraud_Score_Clean'] = pd.to_numeric(df['Fraud_Score'], errors='coerce')
df.loc[df['Fraud_Score_Clean'] < 0, 'Fraud_Score_Clean'] = np.nan
df.drop(columns=['Fraud_Score'], inplace=True)

# 6. SPLIT DELIMITERS
df['Account_Details'] = df['Account_Details'].str.replace('|', '_', regex=False).str.replace('-', '_', regex=False)
df[['Account_Type', 'Account_Status', 'Currency']] = df['Account_Details'].str.split('_', expand=True)
df.drop(columns=['Account_Details'], inplace=True)

# 7. EXPORT TO EXCEL
column_order = [
    'Transaction_Timestamp', 'Customer_ID', 'KYC_Approval_Date_Clean', 
    'Risk_Tier_Clean', 'Fraud_Score_Clean', 'Account_Type', 'Account_Status', 'Currency'
]
df = df[column_order]
df['Transaction_Timestamp'] = df['Transaction_Timestamp'].dt.tz_localize(None)
df['KYC_Approval_Date_Clean'] = df['KYC_Approval_Date_Clean'].dt.tz_localize(None)

df.to_excel('python_cleaned_data.xlsx', index=False, sheet_name='Python_Cleaned_Data')
print("SUCCESS: Master pipeline executed and Excel file created!")

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_11080\2527452990.py:16: UserWarning: Parsing dates in %m/%d/%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(val, dayfirst=True, errors='coerce')


SUCCESS: Master pipeline executed and Excel file created!
